# Проверка гипотез hw01

> Сорочан Дмитрий @legenda0008

In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
SEED = 143

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
INNER_CV_KNN = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)


In [3]:
path = "data/ObesityDataSet_raw_and_data_sinthetic.csv"
df = pd.read_csv(path)

df = df.drop_duplicates()  # см. EDA

print(f"Строк -- {df.shape[0]}, столбцов -- {df.shape[1]}")
df.head()

Строк -- 2087, столбцов -- 17


,Age,Gender,Height,Weight,CALC,FAVC,FCVC,NCP,SCC,SMOKE,CH2O,family_history_with_overweight,FAF,TUE,CAEC,MTRANS,NObeyesdad
0,21.0,Female,1.62,64.0,no,no,2.0,3.0,no,no,2.0,yes,0.0,1.0,Sometimes,Public_Transportation,Normal_Weight
1,21.0,Female,1.52,56.0,Sometimes,no,3.0,3.0,yes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,23.0,Male,1.80,77.0,Frequently,no,2.0,3.0,no,no,2.0,yes,2.0,1.0,Sometimes,Public_Transportation,Normal_Weight
3,27.0,Male,1.80,87.0,Frequently,no,3.0,3.0,no,no,2.0,no,2.0,0.0,Sometimes,Walking,Overweight_Level_I
4,22.0,Male,1.78,89.8,Sometimes,no,2.0,1.0,no,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [4]:
target = "NObeyesdad"

X = df.drop(columns=[target])
y = df[target]

features = X.columns.to_list()
numerical_features = X.select_dtypes("float64").columns.to_list()
str_features = X.select_dtypes("str").columns.to_list()

In [5]:
transformer = ColumnTransformer(
    transformers=[
        (
            "nums",
            StandardScaler(),
            make_column_selector(dtype_include=np.float64),
        ),
        (
            "strs",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include=object),
        ),
    ]
)  # KNN'у нужен скейлер

pipeline = make_pipeline(
    transformer,
    KNeighborsClassifier(),
)


### 1. Рост и вес содержат достаточно информации для классификации

**Гипотеза:** KNN только на `Height` и `Weight` покажет качество не ниже KNN на всех исходных признаках.


In [6]:
df1 = df.copy()

In [7]:
X1 = df1[["Height", "Weight"]]

only2features = cross_val_score(pipeline, X1, y, cv=CV, scoring="f1_macro")
only2features_accuracy = cross_val_score(pipeline, X1, y, cv=CV, scoring="accuracy")

all_features = cross_val_score(pipeline, X, y, cv=CV, scoring="f1_macro")
all_features_accuracy = cross_val_score(pipeline, X, y, cv=CV, scoring="accuracy")

pd.DataFrame(
    {
        "Признаки": ["Height + Weight", "Все признаки"],
        "macro-F1": [only2features.mean(), all_features.mean()],
        "accuracy": [only2features_accuracy.mean(), all_features_accuracy.mean()],
    }
)

,Признаки,macro-F1,accuracy
0,Height + Weight,0.949004,0.950645
1,Все признаки,0.800028,0.823660


> **Вывод:** гипотеза подтвердилась. `Height` и `Weight` дали `macro-F1 = 0.949` и `accuracy = 0.951`, все признаки -- `0.800` и `0.824`.

### 2. Добавление `BMI` и `Weight / Height` повысит качество

**Гипотеза:** новые признаки, отражающие соотношение веса и роста, улучшат качество KNN.


In [8]:
X2 = X.copy()

X2["BMI"] = X2["Weight"] / (X2["Height"]) ** 2
BMIfeatures = cross_val_score(pipeline, X2, y, cv=CV, scoring="f1_macro")
BMIfeatures_accuracy = cross_val_score(pipeline, X2, y, cv=CV, scoring="accuracy")

X2["WH"] = X2["Weight"] / X2["Height"]  # коэффициент наклона облака из EDA
BMIplusWHfeatures = cross_val_score(pipeline, X2, y, cv=CV, scoring="f1_macro")
BMIplusWHfeatures_accuracy = cross_val_score(
    pipeline,
    X2,
    y,
    cv=CV,
    scoring="accuracy",
)

pd.DataFrame(
    {
        "Признаки": ["Исходные", "+ BMI", "+ BMI + Weight / Height"],
        "macro-F1": [
            all_features.mean(),
            BMIfeatures.mean(),
            BMIplusWHfeatures.mean(),
        ],
        "accuracy": [
            all_features_accuracy.mean(),
            BMIfeatures_accuracy.mean(),
            BMIplusWHfeatures_accuracy.mean(),
        ],
    }
)

,Признаки,macro-F1,accuracy
0,Исходные,0.800028,0.823660
1,+ BMI,0.835074,0.853846
2,+ BMI + Weight / Height,0.857072,0.872058


> **Вывод:** гипотеза подтвердилась. `BMI` повысил `macro-F1` с `0.800` до `0.835`, а добавление обоих признаков -- до `0.857`. `Accuracy` выросла с `0.824` до `0.872`.

### 3. `Weight`, `Height` и `BMI` окажутся наиболее важными признаками

**Гипотеза:** антропометрические признаки займут верхние позиции по permutation importance дерева решений на отдельной validation-выборке.

In [9]:
X3 = X.copy()
X3["BMI"] = X3["Weight"] / (X3["Height"]) ** 2

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

X_fit3, X_valid3, y_fit3, y_valid3 = train_test_split(
    X_train3,
    y_train3,
    test_size=0.25,
    stratify=y_train3,
    random_state=SEED,
)

decision_tree = make_pipeline(
    transformer,
    DecisionTreeClassifier(random_state=SEED),
)
decision_tree.fit(X_fit3, y_fit3)

permutation = permutation_importance(
    decision_tree,
    X_valid3,
    y_valid3,
    scoring="f1_macro",
    n_repeats=10,
    random_state=SEED,
)

importances = pd.DataFrame(
    {
        "Признак": X3.columns,
        "Важность": permutation.importances_mean,
    }
).sort_values("Важность", ascending=False)

most_important = importances.head(5)["Признак"].to_list()

importances.head(10).reset_index(drop=True)

,Признак,Важность
0,BMI,0.745406
1,Gender,0.131470
2,Weight,0.119673
3,NCP,0.041614
4,FAVC,0.002512
5,FCVC,0.001906
6,family_history_with_overweight,0.000000
7,MTRANS,0.000000
8,CAEC,0.000000
9,TUE,0.000000


> **Вывод:** гипотеза подтвердилась частично. `BMI` и `Weight` вошли в тройку наиболее важных признаков, однако `Height` не попал в топ-10 по permutation importance на выбранной validation-выборке.

### 4. Пяти лучших признаков будет достаточно для сохранения качества

**Гипотеза:** KNN на пяти лучших признаках не уступит KNN на полном наборе с добавленным `BMI` на отдельной test-выборке.

In [10]:
full_model = clone(pipeline)
most_important_model = clone(pipeline)

full_model.fit(X_train3, y_train3)
most_important_model.fit(X_train3[most_important], y_train3)

full_predictions = full_model.predict(X_test3)
most_important_predictions = most_important_model.predict(X_test3[most_important])

pd.DataFrame(
    {
        "Признаки": ["Все признаки + BMI", "Топ-5"],
        "macro-F1": [
            f1_score(y_test3, full_predictions, average="macro"),
            f1_score(y_test3, most_important_predictions, average="macro"),
        ],
        "accuracy": [
            accuracy_score(y_test3, full_predictions),
            accuracy_score(y_test3, most_important_predictions),
        ],
    }
)

,Признаки,macro-F1,accuracy
0,Все признаки + BMI,0.833542,0.852490
1,Топ-5,0.950747,0.952107


> **Вывод:** гипотеза подтвердилась. На отдельной test-выборке топ-5 повысил `macro-F1` с `0.834` до `0.951` и `accuracy` с `0.852` до `0.952`.

### 5. Удаление почти константных признаков не ухудшит качество

**Гипотеза:** удаление `SMOKE` и `SCC`, у которых одна категория преобладает более чем в 90% строк, не ухудшит или немного повысит качество.


In [11]:
X5 = X.drop(columns=["SCC", "SMOKE"])

noncontstantfeatures = cross_val_score(
    pipeline,
    X5,
    y,
    cv=CV,
    scoring="f1_macro",
)
noncontstantfeatures_accuracy = cross_val_score(
    pipeline,
    X5,
    y,
    cv=CV,
    scoring="accuracy",
)

pd.DataFrame(
    {
        "Признаки": ["Все признаки", "Без SMOKE и SCC"],
        "macro-F1": [all_features.mean(), noncontstantfeatures.mean()],
        "accuracy": [
            all_features_accuracy.mean(),
            noncontstantfeatures_accuracy.mean(),
        ],
    }
)

,Признаки,macro-F1,accuracy
0,Все признаки,0.800028,0.823660
1,Без SMOKE и SCC,0.802154,0.824621


> **Вывод:** заметного эффекта нет. После удаления `SMOKE` и `SCC` `macro-F1` изменился с `0.800` до `0.802`, а `accuracy` -- с `0.824` до `0.825`.

### 6. Регуляризация не даст существенного прироста

**Гипотеза:** при низком числе обусловленности L1- и L2-регуляризация не улучшат логистическую регрессию, обученную практически без регуляризации.

Для чистого сравнения во всех трех моделях используется один solver `saga`. В baseline задано `C = 1e6`, а для L1 и L2 -- `C = 1`.

In [12]:
X6 = X.copy()

pipeline_no_regularization = make_pipeline(
    transformer,
    LogisticRegression(
        C=1e6,
        l1_ratio=0,
        solver="saga",
        max_iter=5000,
        random_state=SEED,
    ),
)
pipeline_l1 = make_pipeline(
    transformer,
    LogisticRegression(
        C=1,
        l1_ratio=1,
        solver="saga",
        max_iter=5000,
        random_state=SEED,
    ),
)
pipeline_l2 = make_pipeline(
    transformer,
    LogisticRegression(
        C=1,
        l1_ratio=0,
        solver="saga",
        max_iter=5000,
        random_state=SEED,
    ),
)

condition_number = np.linalg.cond(
    StandardScaler().fit_transform(X6[numerical_features])
)

no_regularization = cross_val_score(
    pipeline_no_regularization,
    X6,
    y,
    cv=CV,
    scoring="f1_macro",
)
no_regularization_accuracy = cross_val_score(
    pipeline_no_regularization,
    X6,
    y,
    cv=CV,
    scoring="accuracy",
)

l1 = cross_val_score(pipeline_l1, X6, y, cv=CV, scoring="f1_macro")
l1_accuracy = cross_val_score(
    pipeline_l1,
    X6,
    y,
    cv=CV,
    scoring="accuracy",
)

l2 = cross_val_score(pipeline_l2, X6, y, cv=CV, scoring="f1_macro")
l2_accuracy = cross_val_score(
    pipeline_l2,
    X6,
    y,
    cv=CV,
    scoring="accuracy",
)

print(
    f"Число обусловленности: {condition_number} -- и так хорошая обусловленность, улетать в космос параметры, на мой взгляд, не должны, регуляризация вряд ли сильно поможет:"
)

pd.DataFrame(
    {
        "Модель": [
            "Без регуляризации",
            "L1-регуляризация",
            "L2-регуляризация",
        ],
        "macro-F1": [
            no_regularization.mean(),
            l1.mean(),
            l2.mean(),
        ],
        "accuracy": [
            no_regularization_accuracy.mean(),
            l1_accuracy.mean(),
            l2_accuracy.mean(),
        ],
    }
)

Число обусловленности: 2.2214657122491257 -- и так хорошая обусловленность, улетать в космос параметры, на мой взгляд, не должны, регуляризация вряд ли сильно поможет:


,Модель,macro-F1,accuracy
0,Без регуляризации,0.961221,0.962630
1,L1-регуляризация,0.960804,0.962148
2,L2-регуляризация,0.881251,0.885955


> **Вывод:** гипотеза подтвердилась. При числе обусловленности `2.22` L1 практически не изменила качество: `macro-F1` снизился с `0.9612` до `0.9608`. L2 показала заметно худший результат -- `0.8813`.

### 7. Сохранение потенциальных выбросов даст лучшее качество

**Гипотеза:** удаление IQR-выбросов из train приведет к потере полезной информации и снижению качества на неизмененном test.


In [13]:
X7 = X.copy()

X_train7, X_test7, y_train7, y_test7 = train_test_split(
    X7,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

outlier_features = ["Age", "Height", "Weight"]

q1 = X_train7[outlier_features].quantile(0.25)
q3 = X_train7[outlier_features].quantile(0.75)
iqr = q3 - q1

lower_bounds = q1 - 1.5 * iqr
upper_bounds = q3 + 1.5 * iqr

train_outlier_mask = (
    (X_train7[outlier_features] < lower_bounds)
    | (X_train7[outlier_features] > upper_bounds)
).any(axis=1)

X_train7_cleaned = X_train7.loc[~train_outlier_mask]
y_train7_cleaned = y_train7.loc[~train_outlier_mask]

baseline_model = clone(pipeline)
cleaned_model = clone(pipeline)

baseline_model.fit(X_train7, y_train7)
cleaned_model.fit(X_train7_cleaned, y_train7_cleaned)

baseline_predictions = baseline_model.predict(X_test7)
cleaned_predictions = cleaned_model.predict(X_test7)

pd.DataFrame(
    {
        "Вариант": [
            "Без удаления выбросов",
            "С удалением выбросов из train",
        ],
        "Удалено из train": [0, train_outlier_mask.sum()],
        "macro-F1": [
            f1_score(y_test7, baseline_predictions, average="macro"),
            f1_score(y_test7, cleaned_predictions, average="macro"),
        ],
        "accuracy": [
            accuracy_score(y_test7, baseline_predictions),
            accuracy_score(y_test7, cleaned_predictions),
        ],
    }
)

,Вариант,Удалено из train,macro-F1,accuracy
0,Без удаления выбросов,0,0.787555,0.814176
1,С удалением выбросов из train,118,0.753466,0.773946


> **Вывод:** гипотеза подтвердилась. После удаления 118 объектов из train `macro-F1` снизился с `0.788` до `0.753`, `accuracy` -- с `0.814` до `0.774`.

### 8. Подбор гиперпараметров улучшит KNN

**Гипотеза:** KNN с подобранными `n_neighbors` и `weights` покажет более высокий `macro-F1`, чем KNN с параметрами по умолчанию.

Используется вложенная кросс-валидация. На каждом внешнем фолде обычный KNN и `GridSearchCV` обучаются только на outer-train. Подбор параметров происходит во внутренней CV, а итоговая метрика считается на outer-validation, который не участвовал в подборе.

In [14]:
knn_param_grid = {
    "kneighborsclassifier__n_neighbors": [3, 5, 7, 9, 11],
    "kneighborsclassifier__weights": ["uniform", "distance"],
}

knn_nested_rows = []
knn_best_params = []

for fold, (train_idx, valid_idx) in enumerate(CV.split(X, y), start=1):
    X_train_fold = X.iloc[train_idx]
    X_valid_fold = X.iloc[valid_idx]
    y_train_fold = y.iloc[train_idx]
    y_valid_fold = y.iloc[valid_idx]

    default_knn = clone(pipeline)
    default_knn.fit(X_train_fold, y_train_fold)
    default_predictions = default_knn.predict(X_valid_fold)

    search = GridSearchCV(
        estimator=clone(pipeline),
        param_grid=knn_param_grid,
        cv=INNER_CV_KNN,
        scoring="f1_macro",
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train_fold, y_train_fold)
    tuned_predictions = search.predict(X_valid_fold)

    knn_nested_rows.append(
        {
            "fold": fold,
            "default_macro_f1": f1_score(
                y_valid_fold, default_predictions, average="macro"
            ),
            "default_accuracy": accuracy_score(
                y_valid_fold, default_predictions
            ),
            "tuned_macro_f1": f1_score(
                y_valid_fold, tuned_predictions, average="macro"
            ),
            "tuned_accuracy": accuracy_score(
                y_valid_fold, tuned_predictions
            ),
        }
    )
    knn_best_params.append(search.best_params_)

knn_nested_results = pd.DataFrame(knn_nested_rows)

pd.DataFrame(
    {
        "Модель": ["KNN по умолчанию", "KNN после подбора"],
        "macro-F1 mean": [
            knn_nested_results["default_macro_f1"].mean(),
            knn_nested_results["tuned_macro_f1"].mean(),
        ],
        "macro-F1 std": [
            knn_nested_results["default_macro_f1"].std(),
            knn_nested_results["tuned_macro_f1"].std(),
        ],
        "accuracy mean": [
            knn_nested_results["default_accuracy"].mean(),
            knn_nested_results["tuned_accuracy"].mean(),
        ],
        "accuracy std": [
            knn_nested_results["default_accuracy"].std(),
            knn_nested_results["tuned_accuracy"].std(),
        ],
    }
)

,Модель,macro-F1 mean,macro-F1 std,accuracy mean,accuracy std
0,KNN по умолчанию,0.800028,0.013559,0.823660,0.012476
1,KNN после подбора,0.828211,0.016132,0.848108,0.014914


> **Вывод:** гипотеза подтвердилась. Вложенная CV показала рост `macro-F1` с `0.800` до `0.828`, а `accuracy` -- с `0.824` до `0.848`. На каждом внешнем фолде лучшими оказались `n_neighbors = 3` и `weights = "distance"`. Внешние validation-фолды не участвовали в подборе.